In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import joblib

In [2]:
PATIENT_IDS = ["LIB193263", "LIB193264", "LIB193265", "LIB193266", "LIB193267", "LIB193268"]
MAX_GAP = 120  # minutes
LOOKBACK = 24
HORIZON = 4

In [9]:
# 2. LOAD & PREPARE ALL PATIENTS

cgm_all = pd.read_csv("Glucose_measurements.csv")
cgm_all = cgm_all[cgm_all["Patient_ID"].isin(PATIENT_IDS)].copy()

print(f"Total rows for selected patients: {len(cgm_all)}")
print(f"Patients: {cgm_all['Patient_ID'].unique()}")

# Create timestamp
cgm_all["Timestamp"] = pd.to_datetime(
    cgm_all["Measurement_date"] + " " + cgm_all["Measurement_time"]
)

# Sort by patient then timestamp
cgm_all = cgm_all.sort_values(["Patient_ID", "Timestamp"]).reset_index(drop=True)

# Store processed data for each patient separately
patient_data = {}

Total rows for selected patients: 285351
Patients: ['LIB193263' 'LIB193264' 'LIB193265' 'LIB193266' 'LIB193267' 'LIB193268']


In [10]:
# 3. PROCESS EACH PATIENT SEPARATELY

for patient_id in PATIENT_IDS:
    print(f"\n{'='*40}")
    print(f"Processing patient: {patient_id}")
    print(f"{'='*40}")

    # Get patient data
    patient_df = cgm_all[cgm_all["Patient_ID"] == patient_id].copy()

    # Remove duplicates (average if same timestamp)
    patient_df = patient_df.groupby("Timestamp", as_index=False)["Measurement"].mean()

    print(f"Rows after deduplication: {len(patient_df)}")

    # Calculate time gaps
    patient_df["dt_minutes"] = patient_df["Timestamp"].diff().dt.total_seconds() / 60

    # Mark gaps > MAX_GAP
    patient_df["gap"] = patient_df["dt_minutes"] > MAX_GAP

    print(f"Number of gaps > {MAX_GAP} min: {patient_df['gap'].sum()}")
    print(f"Glucose range: {patient_df['Measurement'].min():.1f} - {patient_df['Measurement'].max():.1f}")

    # Keep only needed columns
    patient_df = patient_df[["Timestamp", "Measurement", "gap"]]

    # Store for later use
    patient_data[patient_id] = {
        'df': patient_df,
        'values': patient_df["Measurement"].values.reshape(-1, 1),
        'gaps': patient_df["gap"].values
    }



Processing patient: LIB193263
Rows after deduplication: 60069
Number of gaps > 120 min: 37
Glucose range: 40.0 - 492.0

Processing patient: LIB193264
Rows after deduplication: 26787
Number of gaps > 120 min: 137
Glucose range: 40.0 - 372.0

Processing patient: LIB193265
Rows after deduplication: 46564
Number of gaps > 120 min: 302
Glucose range: 40.0 - 500.0

Processing patient: LIB193266
Rows after deduplication: 55280
Number of gaps > 120 min: 102
Glucose range: 40.0 - 500.0

Processing patient: LIB193267
Rows after deduplication: 55403
Number of gaps > 120 min: 322
Glucose range: 40.0 - 500.0

Processing patient: LIB193268
Rows after deduplication: 41073
Number of gaps > 120 min: 329
Glucose range: 40.0 - 442.0


In [13]:
# 4. SPLIT DATA PATIENT-WISE

# Chronological split per patient, then combine
train_data, val_data, test_data = [], [], []
train_gaps, val_gaps, test_gaps = [], [], []

for patient_id in PATIENT_IDS:
    data = patient_data[patient_id]
    values = data['values']
    gaps = data['gaps']
    n = len(values)

    # Split indices
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    # Append to lists
    train_data.append(values[:train_end])
    val_data.append(values[train_end:val_end])
    test_data.append(values[val_end:])

    train_gaps.append(gaps[:train_end])
    val_gaps.append(gaps[train_end:val_end])
    test_gaps.append(gaps[val_end:])

    print(f"{patient_id}: Train={train_end}, Val={val_end-train_end}, Test={n-val_end}")

# Concatenate all patients
train_raw = np.vstack(train_data)
val_raw = np.vstack(val_data)
test_raw = np.vstack(test_data)

train_gaps_all = np.concatenate(train_gaps)
val_gaps_all = np.concatenate(val_gaps)
test_gaps_all = np.concatenate(test_gaps)

print(f"\nCombined sizes:")
print(f"Train: {len(train_raw)} samples")
print(f"Val: {len(val_raw)} samples")
print(f"Test: {len(test_raw)} samples")

LIB193263: Train=42048, Val=9010, Test=9011
LIB193264: Train=18750, Val=4018, Test=4019
LIB193265: Train=32594, Val=6985, Test=6985
LIB193266: Train=38696, Val=8292, Test=8292
LIB193267: Train=38782, Val=8310, Test=8311
LIB193268: Train=28751, Val=6161, Test=6161

Combined sizes:
Train: 199621 samples
Val: 42776 samples
Test: 42779 samples


In [14]:
# 5. SCALING
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_raw)

train_scaled = scaler.transform(train_raw)
val_scaled = scaler.transform(val_raw)
test_scaled = scaler.transform(test_raw)

print(f"Train min/max: {train_scaled.min():.3f} / {train_scaled.max():.3f}")
print(f"Val min/max: {val_scaled.min():.3f} / {val_scaled.max():.3f}")
print(f"Test min/max: {test_scaled.min():.3f} / {test_scaled.max():.3f}")

# Check for data leakage
val_out_of_bounds = np.any((val_scaled < 0) | (val_scaled > 1))
test_out_of_bounds = np.any((test_scaled < 0) | (test_scaled > 1))
print(f"\nVal outside [0,1]: {val_out_of_bounds}")
print(f"Test outside [0,1]: {test_out_of_bounds}")

Train min/max: 0.000 / 1.000
Val min/max: 0.000 / 0.974
Test min/max: 0.000 / 0.987

Val outside [0,1]: False
Test outside [0,1]: False


In [15]:
# 6. CREATE WINDOWS

def make_windows(series, gaps, lookback, horizon):
    """Create windows while respecting gaps for ALL patients"""
    X, y = [], []

    for i in range(len(series) - lookback - horizon + 1):
        # Skip windows that cross gaps
        if gaps[i:i+lookback+horizon].any():
            continue

        x_seq = series[i:i+lookback, 0]
        y_seq = series[i+lookback:i+lookback+horizon, 0]

        X.append(x_seq)
        y.append(y_seq)

    return np.array(X), np.array(y)


X_train, y_train = make_windows(train_scaled, train_gaps_all, LOOKBACK, HORIZON)
X_val, y_val = make_windows(val_scaled, val_gaps_all, LOOKBACK, HORIZON)
X_test, y_test = make_windows(test_scaled, test_gaps_all, LOOKBACK, HORIZON)

print(f"Train windows: {X_train.shape}")
print(f"Val windows: {X_val.shape}")
print(f"Test windows: {X_test.shape}")


Train windows: (176485, 24)
Val windows: (37594, 24)
Test windows: (37221, 24)


In [16]:
# 7. ADD CHANNEL DIMENSION

X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(f"\nFinal shapes:")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}")


Final shapes:
X_train: (176485, 24, 1)
y_train: (176485, 4)
X_val: (37594, 24, 1)
y_val: (37594, 4)
X_test: (37221, 24, 1)


In [17]:
# 8. SAVE DATA

# Save arrays
np.save('X_train_multi.npy', X_train)
np.save('y_train_multi.npy', y_train)
np.save('X_val_multi.npy', X_val)
np.save('y_val_multi.npy', y_val)
np.save('X_test_multi.npy', X_test)
np.save('y_test_multi.npy', y_test)

# Save scaler
joblib.dump(scaler, 'scaler_multi.pkl')

# Save patient info for reference
patient_info = {
    'patient_ids': PATIENT_IDS,
    'train_raw_sizes': [len(d) for d in train_data],
    'val_raw_sizes': [len(d) for d in val_data],
    'test_raw_sizes': [len(d) for d in test_data],
    'config': {
        'lookback': LOOKBACK,
        'horizon': HORIZON,
        'max_gap': MAX_GAP
    }
}